# Phase 2: Synthetic Cohort Generation and Noise Robustness
Student: Louis Le Forestier and Team
Class: ESILV A4 DIA 3
Course: Machine Learning Final Project


## 1. Project Context and Objectives

In Phase 1, we evaluated simple models on the official CDC Diabetes Health Indicators dataset.
In Phase 2, our goal is to build a controlled synthetic benchmark from scratch:
1. Define a known ground-truth logistic equation linking demographic and lifestyle indicators to diabetes risk.
2. Generate a synthetic cohort of individuals sampled from realistic population distributions.
3. Inject controlled survey noise: self-reporting bias (e.g. under-reporting of BMI), uninformative questionnaire noise features, and minor label noise.
4. Evaluate whether simple interpretable models (Logistic Regression, Decision Trees) can accurately recover the true underlying coefficients and discard noise.


## 2. Environment Setup

We import standard numerical and machine learning packages.
We set a fixed random seed to ensure deterministic reproducibility.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

# Fixed seed for reproducibility
RANDOM_SEED = 42
rng = np.random.RandomState(RANDOM_SEED)


## 3. Step 1: Mathematical Ground-Truth Risk Definition

We define the true latent log-odds equation:
logit(p) = beta_0 + sum beta_j * X_j

The true probability of diabetes is given by the standard sigmoid:
p = 1 / (1 + exp(-logit(p)))

Below are the known ground-truth weights based on established epidemiological associations:
- Intercept: -4.80 (calibrating baseline risk to approximately 14%)
- BMI: +0.08 per unit above 25
- High Blood Pressure: +0.75
- High Cholesterol: +0.55
- General Health: +0.45 per scale step
- Age Category: +0.15 per band
- Physical Activity: -0.30 (protective)
- Smoker: +0.20
- Heavy Alcohol Consumption: -0.15
- Fruit & Vegetable Consumption: -0.10 each


In [ ]:
TRUE_BETAS = {
    "intercept": -4.80,
    "BMI": 0.08,
    "HighBP": 0.75,
    "HighChol": 0.55,
    "GenHlth": 0.45,
    "Age": 0.15,
    "PhysActivity": -0.30,
    "Smoker": 0.20,
    "HvyAlcoholConsump": -0.15,
    "Fruits": -0.10,
    "Veggies": -0.10
}

print("Ground-truth beta coefficients:")
for k, v in TRUE_BETAS.items():
    print(f"  {k:<20}: {v:+.2f}")


## 4. Step 2: Generating the Base Synthetic Population

We generate N = 50,000 synthetic patient profiles.
Variables follow realistic demographic and lifestyle distributions.


In [ ]:
def generate_clean_cohort(n_samples=50000, seed=RANDOM_SEED):
    r = np.random.RandomState(seed)
    
    # Demographics
    age = r.choice(np.arange(1, 14), size=n_samples, p=[
        0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.11, 0.11, 0.10, 0.09, 0.06, 0.05, 0.03
    ])
    sex = r.binomial(1, 0.48, size=n_samples)
    
    # Body Mass Index (normal distribution clipped between 15 and 55)
    bmi = np.clip(r.normal(loc=27.5, scale=5.5, size=n_samples), 15.0, 55.0)
    
    # Lifestyle indicators
    phys = r.binomial(1, 0.75, size=n_samples)
    fruits = r.binomial(1, 0.60, size=n_samples)
    veggies = r.binomial(1, 0.80, size=n_samples)
    smoker = r.binomial(1, 0.40, size=n_samples)
    alcohol = r.binomial(1, 0.06, size=n_samples)
    
    # Correlated clinical indicators
    prob_bp = 1 / (1 + np.exp(-( -4.0 + 0.10 * bmi + 0.25 * age )))
    high_bp = r.binomial(1, prob_bp)
    
    prob_chol = 1 / (1 + np.exp(-( -3.0 + 0.06 * bmi + 0.20 * age )))
    high_chol = r.binomial(1, prob_chol)
    
    gen_hlth = np.clip(np.round(1.5 + 0.05 * bmi + 0.15 * age + 0.5 * high_bp + r.normal(0, 0.5, n_samples)), 1, 5).astype(int)
    
    # Compute true logit and probabilities
    logit = (
        TRUE_BETAS["intercept"]
        + TRUE_BETAS["BMI"] * (bmi - 25.0)
        + TRUE_BETAS["HighBP"] * high_bp
        + TRUE_BETAS["HighChol"] * high_chol
        + TRUE_BETAS["GenHlth"] * (gen_hlth - 1)
        + TRUE_BETAS["Age"] * (age - 1)
        + TRUE_BETAS["PhysActivity"] * phys
        + TRUE_BETAS["Smoker"] * smoker
        + TRUE_BETAS["HvyAlcoholConsump"] * alcohol
        + TRUE_BETAS["Fruits"] * fruits
        + TRUE_BETAS["Veggies"] * veggies
    )
    
    prob = 1.0 / (1.0 + np.exp(-logit))
    y = r.binomial(1, prob)
    
    df = pd.DataFrame({
        "Age": age,
        "Sex": sex,
        "BMI": bmi,
        "HighBP": high_bp,
        "HighChol": high_chol,
        "GenHlth": gen_hlth,
        "PhysActivity": phys,
        "Smoker": smoker,
        "HvyAlcoholConsump": alcohol,
        "Fruits": fruits,
        "Veggies": veggies,
        "True_Probability": prob,
        "Diabetes_Status": y
    })
    return df

df_clean = generate_clean_cohort(n_samples=50000)
print("Synthetic cohort shape:", df_clean.shape)
print("Observed diabetes prevalence:", df_clean["Diabetes_Status"].mean())


## 5. Step 3: Controlled Noise Injection Protocol

We implement three types of realistic survey distortions:
1. Self-reporting bias: overweight individuals systematically under-report BMI by 0.5 to 3.0 points.
2. Stigma under-reporting: a fraction of smokers report being non-smokers.
3. Pure noise questions: we append 10 to 20 completely random features (Gaussian and Bernoulli noise), mirroring Exercise 2 of Lab 4.


In [ ]:
def inject_noise(df, n_noise_features=15, seed=RANDOM_SEED):
    r = np.random.RandomState(seed)
    df_noisy = df.copy()
    n = len(df_noisy)
    
    # 1. Under-reported BMI
    overweight = df_noisy["BMI"] > 27.0
    bmi_error = np.zeros(n)
    bmi_error[overweight] = r.uniform(0.5, 3.0, size=np.sum(overweight))
    df_noisy["BMI_reported"] = np.round(df_noisy["BMI"] - bmi_error, 1)
    
    # 2. Under-reported smoking
    smokers = df_noisy["Smoker"] == 1
    flip = r.binomial(1, 0.15, size=np.sum(smokers))
    smokers_val = df_noisy.loc[smokers, "Smoker"].values
    smokers_val[flip == 1] = 0
    df_noisy.loc[smokers, "Smoker_reported"] = smokers_val
    df_noisy["Smoker_reported"] = df_noisy["Smoker_reported"].fillna(df_noisy["Smoker"]).astype(int)
    
    # 3. Add uninformative noise questions
    for i in range(1, n_noise_features + 1):
        if i % 2 == 0:
            df_noisy[f"Noise_Q{i}"] = r.normal(0, 1, size=n)
        else:
            df_noisy[f"Noise_Q{i}"] = r.binomial(1, 0.5, size=n)
            
    return df_noisy

df_noisy = inject_noise(df_clean, n_noise_features=10)
print("Noisy dataset columns count:", df_noisy.shape[1])


## 6. Step 4: Model Parameter Recovery and Robustness Plan

To evaluate model resilience, we run the following experiments:
1. Benchmark fit: Train Logistic Regression on uncorrupted clean data and verify that estimated coefficients match the true beta values.
2. Noise fit: Train Logistic Regression and L1-penalized Lasso on the noisy data. Check whether L1 regularization correctly assigns zero weights to all uninformative noise questions.
3. Calibration check: Compare predicted probabilities against the true underlying risk probabilities using Brier score and calibration curves.
4. Decision tree comparison: Train a shallow decision tree (depth 3 to 5) and inspect whether top splits isolate the true primary risk factors (HighBP, BMI, GenHlth).


In [ ]:
# Starter test: fitting unregularized logistic regression on clean data
feature_cols = [
    "BMI", "HighBP", "HighChol", "GenHlth", "Age", 
    "PhysActivity", "Smoker", "HvyAlcoholConsump", "Fruits", "Veggies"
]

X_clean = df_clean[feature_cols]
y_clean = df_clean["Diabetes_Status"]

# Standardized logistic regression
pipe_test = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
])

pipe_test.fit(X_clean, y_clean)
print("Model trained successfully on clean synthetic cohort.")
print("ROC-AUC score on clean data:", roc_auc_score(y_clean, pipe_test.predict_proba(X_clean)[:, 1]))


## 7. Next Steps and Work Plan for the Team

- Week 2: Implement parameter recovery metrics comparing estimated weights with true beta parameters.
- Week 3: Experiment with varying noise levels (0%, 5%, 10%, 20% measurement and label error) and plot degradation curves.
- Week 4: Finalize the comparison between the UCI CDC real-world benchmark and the controlled synthetic dataset.
